# Figure 1 assets (real Scenario 23 frames)
CPU is enough. Add the `deepsense6g-scenario-23` dataset, Run All, then download the `fig1` folder from Output.

In [ ]:
import os, glob, numpy as np, pandas as pd, cv2
from scipy import ndimage
OUT = "/kaggle/working/fig1"; os.makedirs(OUT, exist_ok=True)

CSV = [p for p in glob.glob("/kaggle/input/**/scenario23.csv", recursive=True)][0]
BASE = os.path.dirname(CSV); df = pd.read_csv(CSV)
df = df.sort_values(["seq_index", "index"], kind="stable").reset_index(drop=True)
def rp(rel): return os.path.join(BASE, str(rel).lstrip("./"))
seq = df["seq_index"].values
P64 = np.stack([np.loadtxt(rp(v)).ravel() for v in df["unit1_pwr_60ghz"]])
P = P64[:, ::2]; best = P.argmax(1)

# choose a long flight with a wide beam range
cand = []
for s in np.unique(seq):
    ids = np.where(seq == s)[0]
    if len(ids) >= 300: cand.append((np.ptp(best[ids]), len(ids), s))
cand.sort(reverse=True); S = cand[0][2]; ids = np.where(seq == S)[0]
print("flight", S, "length", len(ids), "beam range", cand[0][0])

# background subtraction exactly as in the paper (480x270 grey, +-150 frames median of every 5th frame)
gray = np.stack([cv2.cvtColor(cv2.imread(rp(df.loc[i, "unit1_rgb"]), cv2.IMREAD_REDUCED_COLOR_2), cv2.COLOR_BGR2GRAY) for i in ids]).astype(np.float32)
L, H, W = gray.shape; cent = np.full((L, 3), np.nan); prev = None; diffs = {}
for c0 in range(0, L, 25):
    lo, hi = max(0, c0 - 150), min(L, c0 + 175); bg = np.median(gray[lo:hi:5], 0)
    for j in range(c0, min(L, c0 + 25)):
        d = cv2.GaussianBlur(np.abs(gray[j] - bg), (5, 5), 0); diffs[j] = d
        lab, n = ndimage.label(d > max(d.mean() + 5 * d.std(), 8.0))
        if n == 0: continue
        idx = np.arange(1, n + 1); area = np.asarray(ndimage.sum(np.ones_like(d), lab, idx)); mass = np.asarray(ndimage.sum(d, lab, idx))
        ok = np.where((area >= 2) & (area <= 0.003 * H * W))[0]
        if len(ok) == 0: continue
        com = np.array(ndimage.center_of_mass(d, lab, idx[ok])); cx, cy = com[:, 1] / W, com[:, 0] / H
        sc = mass[ok] if prev is None else mass[ok] / (1 + 20 * np.hypot(cx - prev[0], cy - prev[1]))
        k = int(np.argmax(sc)); cent[j] = [cx[k], cy[k], area[ok][k]]; prev = (cx[k], cy[k])

# three frames with low / middle / high optimal beam and a valid detection
valid = np.where(~np.isnan(cent[:, 0]))[0]
bb = best[ids][valid]; picks = []
for q in (0.1, 0.5, 0.9):
    target = np.quantile(bb, q); picks.append(int(valid[np.argmin(np.abs(bb - target))]))
for n, j in enumerate(picks):
    i = ids[j]
    img = cv2.imread(rp(df.loc[i, "unit1_rgb"]), cv2.IMREAD_COLOR)          # original resolution
    cv2.imwrite(f"{OUT}/frame_{n}.png", img)
    d = diffs[j]; d8 = np.clip(d / max(d.max(), 1) * 255, 0, 255).astype(np.uint8)
    cv2.imwrite(f"{OUT}/diff_{n}.png", d8)
np.savez(f"{OUT}/assets.npz", flight=S, picks=np.array(picks), ids=ids, cent=cent, best=best[ids], P64=P64[ids],
         height=df.loc[ids, "unit2_height"].values if "unit2_height" in df else np.zeros(len(ids)))
print("saved", os.listdir(OUT))